# 广东省部分海域监测数据合并

## 程序包初始化

### 内建程序包

In [1]:
import io, sys, os; 
try:
    nb_dir = nb_dir; 
except NameError: 
    nb_dir = os.getcwd(); 
    sys.path.append(nb_dir); 

In [2]:
import re; 
import collections as coll, itertools as it; 
import operator as op; 

In [3]:
import sqlite3; 
from datetime import datetime; 

### 第三方和自定义程序包

In [4]:
import numpy as np, pandas as pd; 

## `xls`读取和预处理

### 路径定位
文件使用相对路径

In [5]:
wq_spot_value_path = os.path.join(
    nb_dir, os.pardir, os.pardir, os.pardir, 
    "Source", "In-Situ_Monitoring"
); 
wq_spot_value_path = os.path.normpath(wq_spot_value_path); 

### 文件名匹配
文件名称格式: `七位数码.xls`; 

据此构建正则表达式, 在相对路径的目录下搜索并匹配符合条件的文件名. 

In [6]:
file_mar_filter = re.compile(
    r"\A[0-9]{7}\.xls\Z"
); 

In [7]:
wq_mar_spot_filename_xls = list(); 
for node, dirs, files in os.walk(wq_spot_value_path): 
    if node is not wq_spot_value_path: 
        continue; 
    for file in files: 
        if not file_mar_filter.match(file): 
            continue; 
        wq_mar_spot_filename_xls.append(os.path.join(node, file)); 

### 数据的标准化规则定义
`sqlite`中, 同一表格内, 不同记录在相同字段的数据类型必须相同. 

为了后续计算的便利, 需要对部分数据进行标准化处理: 
* 唯一识别码 (`id` 字段): 用于在合并多年, 多季节表格后, 保证记录id的唯一性
    > 由三部分组成, 相邻的两部分之间以负号连接
    > 1. 监测站省份代码 (监测站代码前三位), 三位大写字母
    > 2. 实测数据发行批次, 四位发行年份数字 + 一位发行季节代号大写字母 (A=春, B=夏, C=秋). 
    > 3. 记录在本次发行数据内的id, 四位数字, 不足位时在左侧补零. 
* 日期 (`dates`字段): 转换为自公元1970年01月01日开始的**天数**. 
    > 1970年1月1日为第0天, 1月2日为第1天, 1971年1月1日为第365天, 以此类推. 
    >
    > 此处不使用`Unix`时间戳, 因为该时间戳精确到秒, 而且需要考虑时区问题, \
    > 但水质取样日期只精确到日, 没有具体的时分秒信息. 
* 水质测量结果中低于检出限的结果处理: 
    * 如果表示结果的文本, 为大于零的整数或小数的字符串形式, 直接按字面量转化为浮点数;  
    * 如果表示结果的文本为"0", 直接转换为浮点数`0.`; 
    * 如果表示结果的文本, 为`L`结尾, 前接一个正整数或正小数的字符串 (例如 "0.002L"), 表示测量结果**低于测定方法的检出限**. 
        * 不能直接其视为`0. ` (因为证据不足, 否则在原始数据中就会直接记为"0"); 
        * 它的含义与实测结果中直接出现的零值不同. 
        * 参考2016年至2025年间发表的部分文献所述的方法, 后续计算过程中**按照检出限的$1 \over 2$处理**$^{[1-5]}$

参考文献: 
```
[1] 朱媛媛, 田进军, 李红亮, 等. 丹江口水库水质评价及水污染特征[J]. 
    农业环境科学学报, 2016, 35(01): 139-147.
    
[2] 蔡竹, 李垚垚, 何丽. 石笋沟水质现状评价[J]. 贵阳学院学报(自然
    科学版), 2020, 15(03): 45-47. 
    
[3] 刘光正, 王明森, 刘健, 等. 大明湖水污染物因子分析及富营养化评价[J]. 
    济南大学学报(自然科学版), 2023, 37(06): 696-702. 
    
[4] 陈振明. 汕头市近岸海域水质状况研究[J]. 黑龙江环境通报, 2024, 
    37(08): 13-15. 

[5] 朱纯祥. 龙河口水库水体富营养化评价及其变化趋势研究[J]. 安徽水利
    水电职业技术学院学报, 2025, 25(03): 28-31+43. 
```


In [8]:
class MarineWaterQualityPreprocessor(object): 
    
    UNIX_EPOCH = datetime(1970, 1, 1); 
    LAT_LON_FILTER = re.compile(
        r"E[:：](?P<lon>[0-9]{1,3}\.+[^，]*)，"
        r"N[:：](?P<lat>[0-9]{1,2}\.+[^，]*)"
    ); 
    LAT_LON_DUPLI_DECIMAL_FILTER = re.compile(
        r"(?P<decimal>\.){2,}"
    ); 
    PROV_GETTER = op.itemgetter(slice(None, 3)); 
    CONC_FLOAT_LITERAL = r"[0-9]+(\.[0-9]*)?([Ee][+-]?[0-9]+)?"; 
    CONC_LESS_SIGN = r"L?"; 
    CONC_LOWER_LIMIT_FILTER = re.compile(
        r"\A(?P<lim>{fp})(?P<lt>{lt})\Z".format(
            lt=CONC_LESS_SIGN, fp=CONC_FLOAT_LITERAL
        )
    ); 
    
    #多表格合并后的新id格式
    
    @classmethod
    def unique_id(cls, sheet, year, prov, id_obsv): 
        return "{prov}-{year}{sheet}-{id:0>4d}".format(
            prov=prov, year=year, sheet="ABC"[sheet], id=id_obsv
        ); 
    
    #将日期 (年-月-日字符串或 excel 日期常量) 统一转换为 Python datetime 对象
    
    @classmethod
    def ymd_to_datetime(cls, ymd): 
        if type(ymd) is str: # 若 excel 日期未被解析, pandas导入结果为原本的年-月-日字符串
            return datetime.fromisoformat(ymd); 
        if type(ymd) is pd.Timestamp: # 若 excel 日期被解析, pandas导入结果为 pd.Timestamp 对象
            return ymd.to_pydatetime(); 
    
    #计算1970年1月1日开始的天数
    
    @classmethod
    def pydatetime_to_days(cls, pydatetime): 
        date_diff = pydatetime - cls.UNIX_EPOCH; 
        days = date_diff.days; 
        return days; 
            
    #提取年, 月, 日构成的三元组
    
    @classmethod
    def pydatetime_to_ymd_trituple(cls, pydatetime): 
        year_obsv = pydatetime.year; 
        month_obsv = pydatetime.month; 
        day_obsv = pydatetime.day; 
        return year_obsv, month_obsv, day_obsv; 

    #经纬度字符串笔误纠正
    
    @classmethod
    def from_lat_lon_errata(cls, coord_text): 
        coord_mod = coord_text; 
        coord_mod = re.sub(cls.LAT_LON_DUPLI_DECIMAL_FILTER, "\g<decimal>", coord_mod); 
        coord_mod = re.sub(",", str(), coord_mod); 
        return coord_mod; 
    
    #解析经纬度
    
    @classmethod
    def lat_lon_parse(cls, geo_text): 
        match = cls.LAT_LON_FILTER.match(geo_text); 
        if match: 
            match_grp = match.groupdict(); 
            lat = match_grp["lat"]; 
            lon = match_grp["lon"]; 
            #纠正数据中的部分笔误, 主要包括重复输入的小数点和逗号
            lat = cls.from_lat_lon_errata(lat); 
            lon = cls.from_lat_lon_errata(lon); 
        else: 
            lat = None; lon = None; 
        return lat, lon; 

    # 解析浓度
    
    @classmethod
    def conc_parse(cls, data): 
        match = cls.CONC_LOWER_LIMIT_FILTER.match(data); 
        if match: 
            match_grp = match.groupdict(); 
            num = np.float64(match_grp["lim"]); 
            if match_grp["lt"]: 
                num /= 2; 
        else: 
            num = None; 
        return num; 

    
    #数据读取与预处理函数主入口
    
    @classmethod
    def read_and_process(cls, file, sheet): 
        #读取xls文件, 形成DataFrame
        df_sheet = pd.read_excel(
            file, sheet_name=sheet, header=None, skiprows=3
        ); 
        #获取省份
        prov = df_sheet[2].map(cls.PROV_GETTER).mode()[0]; 
        #日期数据导入
        ser_datetime = df_sheet[4].map(cls.ymd_to_datetime); 
        ser_ymd_obsv = ser_datetime.map(cls.pydatetime_to_ymd_trituple); 
        ser_year_obsv, ser_month_obsv, ser_day_obsv = (
            pd.Series(col) for col in zip(*ser_ymd_obsv.tolist())
        ); 
        year = ser_year_obsv.mode()[0]; 
        ser_dates = ser_datetime.map(cls.pydatetime_to_days); 
        #构造新id
        ser_id = df_sheet[0].map(
            lambda id: cls.unique_id(sheet, year, prov, id_obsv=id)
        ); 
        #提取经纬度
        ser_lat_lon = df_sheet[3].map(cls.lat_lon_parse); 
        ser_lat, ser_lon = (
            pd.Series(col) for col in zip(*ser_lat_lon.tolist())
        ); 
        #提取浓度
        ser_ph = df_sheet[5].astype(np.str).map(cls.conc_parse); #pH
        ser_in = df_sheet[6].astype(np.str).map(cls.conc_parse); #无机氮
        ser_lp = df_sheet[7].astype(np.str).map(cls.conc_parse); #活性磷酸盐
        ser_pet = df_sheet[8].astype(np.str).map(cls.conc_parse); #石油类
        ser_do = df_sheet[9].astype(np.str).map(cls.conc_parse); #溶解氧
        ser_cod = df_sheet[10].astype(np.str).map(cls.conc_parse); #化学需氧量
        #整理数据
        df_proc = pd.DataFrame.from_dict({
            "id": ser_id.astype(np.str), 
            "dates": ser_dates.astype(np.int), 
            "year_obsv": ser_year_obsv.astype(np.int), 
            "month_obsv": ser_month_obsv.astype(np.int), 
            "day_obsv": ser_day_obsv.astype(np.int), 
            "lon": ser_lon.astype(np.float64), 
            "lat": ser_lat.astype(np.float64), 
            "station": df_sheet[2].astype(np.str), 
            "city": df_sheet[1].astype(np.str), 
            "ph": ser_ph.astype(np.float64), 
            "inorganic_nitrogen": ser_in.astype(np.float64), 
            "labile_phosphate": ser_lp.astype(np.float64), 
            "petroleum": ser_pet.astype(np.float64), 
            "dissolved_oxygen": ser_do.astype(np.float64), 
            "chemical_oxygen_demand": ser_cod.astype(np.float64), 
        } ).set_index("id"); 
        return df_proc; 

### 数据批量读取, 标准化和合并

In [9]:
df_merge = pd.DataFrame(); 
for file, sheet in it.product(wq_mar_spot_filename_xls, range(3)): 
    df = MarineWaterQualityPreprocessor.read_and_process(file, sheet); 
    df_merge = df_merge.append(df); 

## 提取结果建表入库

### 建立数据库连接并创建表格

In [10]:
wq_items_sqlite = sqlite3.connect(os.sep.join(
    [nb_dir, "wq_spot_value.sqlite"]
) ); 

### 数据的插入

In [11]:
df_merge.to_sql(
    name="info_spot_value_marine", con=wq_items_sqlite, 
    if_exists="replace"
); 

### 数据库保存与关闭

In [12]:
wq_items_sqlite.execute("VACUUM"); 
wq_items_sqlite.commit(); 
wq_items_sqlite.close(); 